In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import string

In [ ]:
# 1
data = [
    ("How can I reset my pisword?", "You can reset it through the account settings"),
    ("Where is my order?", "You can track your order in the dashboard"),
    ("What are your working hours?", "We are open from 9 AM to 5 PM"),
    ("How do I contact support?", "Email us at support at example dot com"),
    ("Can I cancel my subscription?", "Yes you can cancel in the billing section"),
    ("Is there a refund policy?", "Yes we offer a 30 day money back guarantee"),
    ("Do you offer international shipping?", "Yes we ship worldwide"),
    ("How to change my email?", "Go to profile settings to update your email for 6777"),
    ("What payment methods do you accept 67?", "We accept credit cards and paypal"),
    ("Is my data secure?", "We use standard encryption to protect your data"),
    ("How to delete my account?", "Please contact support to delete your account")
]

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text


clean_data = [(clean_text(q), clean_text(a)) for q, a in data]

word2index = {"<PAD>": 0, "<EOS>": 1}
index2word = {0: "<PAD>", 1: "<EOS>"}

for q, a in clean_data:
    for word in q.split() + a.split():
        if word not in word2index:
            word2index[word] = len(word2index)
            index2word[len(index2word)] = word

vocab_size = len(word2index)
max_q_len = max(len(q.split()) for q, a in clean_data)
max_a_len = max(len(a.split()) for q, a in clean_data)

print(f"Розмір словника (input_size): {vocab_size}")
print(f"Максимальна довжина вхідної послідовності: {max_q_len}")
print(f"Максимальна довжина вихідної послідовності: {max_a_len}")

max_len = max(max_q_len, max_a_len) + 1


def pad_sequence(sentence, max_length):
    tokens = sentence.split()
    indices = [word2index[word] for word in tokens] + [word2index["<EOS>"]]
    return indices + [word2index["<PAD>"]] * (max_length - len(indices))


X_tensor = torch.tensor([pad_sequence(q, max_len) for q, a in clean_data], dtype=torch.long)
Y_tensor = torch.tensor([pad_sequence(a, max_len) for q, a in clean_data], dtype=torch.long)

Розмір словника (input_size): 83
Максимальна довжина вхідної послідовності (запитання): 7
Максимальна довжина вихідної послідовності (відповіді): 10


In [ ]:
# 2
class SupportRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=128):
        super(SupportRNN, self).__init__()

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)
        
        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        # x розмірність: (batch_size, seq_length)
        embedded = self.embedding(x)          # -> (batch_size, seq_length, vocab_size)
        out, hidden = self.rnn(embedded)      # -> (batch_size, seq_length, hidden_size)
        logits = self.fc(out)                 # -> (batch_size, seq_length, vocab_size)
        return logits

# Ініціалізація моделі, функції втрат та оптимізатора
model = SupportRNN(vocab_size=vocab_size, hidden_size=128)

# Використовуємо CrossEntropyLoss, ігноруючи токени <PAD>, щоб вони не впливали на помилку
criterion = nn.CrossEntropyLoss(ignore_index=word2index["<PAD>"])
optimizer = optim.Adam(model.parameters(), lr=0.001)


# 2. Підготовка тензорів (Padding)
# Для простоти роботи RNN з батчами, вирівнюємо всі речення до однакової максимальної довжини
max_len = max(max_q_len, max_a_len) + 1 # +1 для токена <EOS>

def pad_sequence(sentence, max_length):
    tokens = sentence.split()
    indices = [word2index[word] for word in tokens] + [word2index["<EOS>"]]
    padded = indices + [word2index["<PAD>"]] * (max_length - len(indices))
    return padded

X_data = [pad_sequence(q, max_len) for q, a in clean_data]
Y_data = [pad_sequence(a, max_len) for q, a in clean_data]

X_tensor = torch.tensor(X_data, dtype=torch.long)
Y_tensor = torch.tensor(Y_data, dtype=torch.long)


# (10 епох)
epochs = 10

print("Починаємо навчання...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()   
    
    # Прямий прохід
    outputs = model(X_tensor) # Розмірність: (batch_size, max_len, vocab_size)
    
    # Змінюємо форму тензорів для розрахунку функції втрат
    # CrossEntropyLoss очікує передбачення у вигляді (N, C), а цілі - у вигляді (N)
    outputs_flattened = outputs.view(-1, vocab_size) 
    targets_flattened = Y_tensor.view(-1)
    
    # Розрахунок втрат
    loss = criterion(outputs_flattened, targets_flattened)
    
    # Зворотне поширення помилки та крок оптимізації
    loss.backward()
    optimizer.step()
    
    print(f"Епоха [{epoch+1}/{epochs}], Втрати (Loss): {loss.item():.4f}")

Починаємо навчання...
Епоха [1/10], Втрати (Loss): 4.4628
Епоха [2/10], Втрати (Loss): 4.3374
Епоха [3/10], Втрати (Loss): 4.2160
Епоха [4/10], Втрати (Loss): 4.0987
Епоха [5/10], Втрати (Loss): 3.9855
Епоха [6/10], Втрати (Loss): 3.8761
Епоха [7/10], Втрати (Loss): 3.7705
Епоха [8/10], Втрати (Loss): 3.6686
Епоха [9/10], Втрати (Loss): 3.5703
Епоха [10/10], Втрати (Loss): 3.4754


In [5]:
# 3
class SupportLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size=128):
        super(SupportLSTM, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=vocab_size)
        # Замінили nn.RNN на nn.LSTM
        self.lstm = nn.LSTM(input_size=vocab_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        # LSTM повертає out, а також кортеж (hidden_state, cell_state)
        out, (hidden, cell) = self.lstm(embedded)
        logits = self.fc(out)
        return logits

# Ініціалізація
model_lstm = SupportLSTM(vocab_size=vocab_size, hidden_size=128)
criterion = nn.CrossEntropyLoss(ignore_index=word2index["<PAD>"])
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=0.001)

# 10 епох
print("Починаємо навчання LSTM...")
epochs = 10
for epoch in range(epochs):
    model_lstm.train()
    optimizer_lstm.zero_grad()
    
    outputs = model_lstm(X_tensor)
    
    outputs_flattened = outputs.view(-1, vocab_size) 
    targets_flattened = Y_tensor.view(-1)
    
    loss = criterion(outputs_flattened, targets_flattened)
    loss.backward()
    optimizer_lstm.step()
    
    print(f"Епоха [{epoch+1}/{epochs}], Втрати (Loss): {loss.item():.4f}")

# 3. Генерація відповідей для 5 прикладів
print("\n--- Передбачення моделі LSTM ---")
model_lstm.eval()

# Беремо перші 5 запитань з нашого набору
test_queries = [q for q, a in clean_data[:5]]

for query in test_queries:
    # Підготовка вхідних даних (додаємо паддінг)
    tokens = query.split()
    indices = [word2index.get(word, 0) for word in tokens] + [word2index["<EOS>"]]
    padded_indices = indices + [word2index["<PAD>"]] * (max_len - len(indices))
    
    input_tensor = torch.tensor([padded_indices], dtype=torch.long)
    
    # Отримання передбачень
    with torch.no_grad():
        output = model_lstm(input_tensor) # (1, max_len, vocab_size)
    
    # Знаходимо індекси слів з максимальною ймовірністю
    predicted_indices = torch.argmax(output, dim=-1).squeeze().tolist()
    
    # Конвертуємо індекси назад у слова
    predicted_words = [index2word.get(idx, "") for idx in predicted_indices if idx not in (word2index["<PAD>"], word2index["<EOS>"])]
    predicted_answer = " ".join(predicted_words)
    
    print(f"Запит: {query}")
    print(f"Відповідь: {predicted_answer}\n")

Починаємо навчання LSTM...
Епоха [1/10], Втрати (Loss): 4.3948
Епоха [2/10], Втрати (Loss): 4.3543
Епоха [3/10], Втрати (Loss): 4.3146
Епоха [4/10], Втрати (Loss): 4.2748
Епоха [5/10], Втрати (Loss): 4.2345
Епоха [6/10], Втрати (Loss): 4.1930
Епоха [7/10], Втрати (Loss): 4.1502
Епоха [8/10], Втрати (Loss): 4.1055
Епоха [9/10], Втрати (Loss): 4.0590
Епоха [10/10], Втрати (Loss): 4.0105

--- Передбачення моделі LSTM ---
Запит: how can i reset my password
Відповідь: you can you it to account account

Запит: where is my order
Відповідь: worldwide can to your your

Запит: what are your working hours
Відповідь: payment order open from account day

Запит: how do i contact support
Відповідь: you yes at ship at day

Запит: can i cancel my subscription
Відповідь: can you can to in day



In [7]:
# 4
from transformers import pipeline

print("Завантаження моделі distilgpt2...")
# Використовуємо pipeline для генерації тексту. 
# distilgpt2 - це полегшена версія GPT-2.
generator = pipeline('text-generation', model='distilgpt2')

# ==========================================
# 2. Тестування на коротких запитах
# ==========================================
queries = [
    "I forgot my password, how can I",
    "My order is delayed, what should I do"
]

print("\n--- Передбачення моделі Transformer ---")
for query in queries:
    # Генеруємо текст. max_new_tokens контролює довжину доданого тексту
    result = generator(query, max_new_tokens=15, num_return_sequences=1, truncation=True)
    generated_text = result[0]['generated_text']
    
    print(f"Запит: {query}")
    print(f"Відповідь моделі: {generated_text}\n")

c:\Users\Elizaveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Завантаження моделі distilgpt2...


c:\Users\Elizaveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Elizaveta\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Elizaveta\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\fil


--- Передбачення моделі Transformer ---
Запит: I forgot my password, how can I
Відповідь моделі: I forgot my password, how can I change it?





A lot of people keep saying

Запит: My order is delayed, what should I do
Відповідь моделі: My order is delayed, what should I do?”

The first thing I wanted to do was get the

